# Qualitative Samples of Normative Simulacra

Sample norms, CI flows, and consolidated norms from **1984** and **Pride and Prejudice** for the COLM 2026 paper appendix.

In [3]:
import pandas as pd
import json
import textwrap
from IPython.display import display, Markdown

norms = pd.read_parquet('/share/pierson/matt/n2s4cir/data/fiction10/abstracted_norms.parquet')
flows = pd.read_parquet('/share/pierson/matt/n2s4cir/data/fiction10/ci_flows.parquet')


BOOKS = {
    '1984': '\\textit{1984}',
    '1342': '\\textit{Pride and Prejudice}',
}

print(f'Norms: {len(norms):,}, Flows: {len(flows):,}')

Norms: 11,869, Flows: 1,241


## 1. Sample Norms (Raz Anatomy)
Select 3 diverse norms per book — one each of obligatory, prohibited, and recommended.

In [4]:
def sample_norms(df, gid, n_per_force=1, seed=777):
    """Sample norms with diverse normative force."""
    book = df[df['gutenberg_id'] == gid].copy()
    # Filter to norms with good articulations
    book = book[book['raz_norm_articulation'].str.len() > 20]
    samples = []
    for force in ['obligatory', 'prohibited', 'recommended']:
        sub = book[book['raz_normative_force'] == force]
        if len(sub) > 0:
            samples.append(sub.sample(n=min(n_per_force, len(sub)), random_state=seed))
    return pd.concat(samples) if samples else pd.DataFrame()

norm_samples = {}
for gid in BOOKS:
    norm_samples[gid] = sample_norms(norms, gid)

# Display
for gid, title in BOOKS.items():
    print(f'\n=== {title} ===')
    for _, row in norm_samples[gid].iterrows():
        print(f'  [{row["raz_normative_force"]}] {row["raz_norm_articulation"]}')
        print(f'    Subject: {row["raz_norm_subject"]}')
        print(f'    Act: {row["raz_norm_act"]}')
        print(f'    Condition: {row["raz_condition_of_application"]}')
        print(f'    Context: {row["raz_context"]}')
        print()


=== \textit{1984} ===
  [obligatory] Individuals under continuous state surveillance must remain exactly where they are and make no movement when under the surveillance of the telescreen and the iron voice.
    Subject: individuals under continuous state surveillance whose every action is monitored for compliance
    Act: remain exactly where you are, make no movement
    Condition: when under the surveillance of the telescreen and the iron voice
    Context: surveillance and control

  [prohibited] A member of a totalitarian political organization must not skip mandatory communal activities when expected to attend the Community Centre.
    Subject: a member of a totalitarian political organization
    Act: skip mandatory communal activities
    Condition: when the member is expected to attend the Community Centre
    Context: communal life

  [recommended] Children in a society that emphasizes vigilance and security ought to notice and report unusual or foreign characteristics when t

## 2. Sample CI Information Flows

In [5]:
def sample_flows(df, gid, n=3, seed=42):
    """Sample flows with diverse appropriateness judgments."""
    book = df[df['gutenberg_id'] == gid].copy()
    book = book[book['ci_sender'].str.len() > 2]
    samples = []
    for appr in ['appropriate', 'inappropriate', 'ambiguous']:
        sub = book[book['ci_appropriateness'] == appr]
        if len(sub) > 0:
            samples.append(sub.sample(n=min(1, len(sub)), random_state=seed))
    return pd.concat(samples) if samples else book.sample(n=min(n, len(book)), random_state=seed)

flow_samples = {}
for gid in BOOKS:
    flow_samples[gid] = sample_flows(flows, gid)

for gid, title in BOOKS.items():
    print(f'\n=== {title} ===')
    for _, row in flow_samples[gid].iterrows():
        print(f'  [{row["ci_appropriateness"]}]')
        print(f'    Sender: {row["ci_sender"]}')
        print(f'    Recipient: {row["ci_recipient"]}')
        print(f'    Subject: {row.get("ci_subject", "N/A")}')
        print(f'    Info type: {row["ci_information_type"]}')
        print(f'    Transmission: {row["ci_transmission_principle"]}')
        print(f'    Context: {row["ci_context"]}')
        print(f'    Norms invoked: {row["ci_norms_invoked"]}')
        print()


=== \textit{1984} ===
  [appropriate]
    Sender: O'Brien
    Recipient: Winston
    Subject: Goldstein's book
    Info type: a copy of Goldstein's book
    Transmission: covert and secretive method to avoid detection
    Context: covert operation
    Norms invoked: ["The information must be transmitted in a way that avoids detection and ensures secrecy"]

  [inappropriate]
    Sender: the skull-faced man
    Recipient: the guards and the officer
    Subject: the chinless man
    Info type: accusations or charges against others
    Transmission: prohibition
    Context: legal/judicial
    Norms invoked: ["One must not bear false witness against one's neighbor"]

  [ambiguous]
    Sender: the protagonist
    Recipient: other individuals or the public
    Subject: historical evidence
    Info type: historical evidence
    Transmission: potential resistance against the Party's control
    Context: political resistance
    Norms invoked: ["The protagonist should consider the potential con

## 4. Generate LaTeX for Paper Appendix

In [ ]:
def escape_latex(s):
    """Escape special LaTeX characters."""
    if not isinstance(s, str):
        return str(s)
    for ch in ['&', '%', '$', '#', '_', '{', '}']:
        s = s.replace(ch, '\\' + ch)
    s = s.replace('~', '\\textasciitilde{}')
    s = s.replace('^', '\\textasciicircum{}')
    return s

def wrap(s, width=60):
    return escape_latex(str(s)[:200])

latex_lines = []
latex_lines.append(r'\section{Qualitative Examples of Normative Simulacra}')
latex_lines.append(r'\label{app:qualitative-examples}')
latex_lines.append('')

# --- Norms table ---
latex_lines.append(r'\subsection{Extracted Norms (Raz Anatomy)}')
latex_lines.append(r'\begin{table}[ht]')
latex_lines.append(r'\centering')
latex_lines.append(r'\small')
latex_lines.append(r'\begin{tabular}{p{1.8cm}p{1.2cm}p{2.5cm}p{2.5cm}p{3.5cm}}')
latex_lines.append(r'\toprule')
latex_lines.append(r'Source & Force & Subject & Act & Articulation \\')
latex_lines.append(r'\midrule')

for gid, title_latex in BOOKS.items():
    for _, row in norm_samples[gid].iterrows():
        latex_lines.append(
            f'{title_latex} & {escape_latex(row["raz_normative_force"])} & '
            f'{wrap(row["raz_norm_subject"])} & '
            f'{wrap(row["raz_norm_act"])} & '
            f'{wrap(row["raz_norm_articulation"])} \\\\'
        )

latex_lines.append(r'\bottomrule')
latex_lines.append(r'\end{tabular}')
latex_lines.append(r'\caption{Sample extracted norms from \textit{1984} and \textit{Pride and Prejudice}.}')
latex_lines.append(r'\label{tab:sample-norms}')
latex_lines.append(r'\end{table}')
latex_lines.append('')

# --- Flows table ---
latex_lines.append(r'\subsection{Extracted CI Information Flows}')
latex_lines.append(r'\begin{table}[ht]')
latex_lines.append(r'\centering')
latex_lines.append(r'\small')
latex_lines.append(r'\begin{tabular}{p{1.8cm}p{1.2cm}p{1.8cm}p{1.8cm}p{2cm}p{2.5cm}}')
latex_lines.append(r'\toprule')
latex_lines.append(r'Source & Appr. & Sender & Recipient & Info Type & Trans. Principle \\')
latex_lines.append(r'\midrule')

for gid, title_latex in BOOKS.items():
    for _, row in flow_samples[gid].iterrows():
        appr = row['ci_appropriateness'][:5] + '.' if len(row['ci_appropriateness']) > 5 else row['ci_appropriateness']
        latex_lines.append(
            f'{title_latex} & {escape_latex(appr)} & '
            f'{wrap(row["ci_sender"])} & '
            f'{wrap(row["ci_recipient"])} & '
            f'{wrap(row["ci_information_type"])} & '
            f'{wrap(row["ci_transmission_principle"])} \\\\'
        )

latex_lines.append(r'\bottomrule')
latex_lines.append(r'\end{tabular}')
latex_lines.append(r'\caption{Sample extracted CI information flows from \textit{1984} and \textit{Pride and Prejudice}.}')
latex_lines.append(r'\label{tab:sample-flows}')
latex_lines.append(r'\end{table}')
latex_lines.append('')

# --- Consolidated norms table ---
latex_lines.append(r'\subsection{Consolidated Norms (Normative Universe)}')
latex_lines.append(r'\begin{table}[ht]')
latex_lines.append(r'\centering')
latex_lines.append(r'\small')
latex_lines.append(r'\begin{tabular}{p{1.8cm}p{0.5cm}p{4cm}p{5cm}}')
latex_lines.append(r'\toprule')
latex_lines.append(r'Source & $n$ & Canonical Articulation & Source Norms (sample) \\')
latex_lines.append(r'\midrule')

for gid, title_latex in BOOKS.items():
    for _, row in consol_samples[gid].iterrows():
        sources = json.loads(row['source_norm_articulations'])
        source_sample = '; '.join(s[:80] for s in sources[:3])
        if len(sources) > 3:
            source_sample += f' (+{len(sources)-3} more)'
        latex_lines.append(
            f'{title_latex} & {row["cluster_size"]} & '
            f'{wrap(row["canonical_norm_articulation"])} & '
            f'{wrap(source_sample)} \\\\'
        )

latex_lines.append(r'\bottomrule')
latex_lines.append(r'\end{tabular}')
latex_lines.append(r'\caption{Sample consolidated norms showing cluster merging. $n$: number of source norms merged into each canonical norm.}')
latex_lines.append(r'\label{tab:sample-consolidated}')
latex_lines.append(r'\end{table}')

latex_output = '\n'.join(latex_lines)
print(latex_output)

In [ ]:
# Save LaTeX to file for easy inclusion
out_path = '/share/pierson/matt/papers/colm26_llm-praxis/06_appendix_qualitative.tex'
with open(out_path, 'w') as f:
    f.write(latex_output)
print(f'Saved to {out_path}')
print('Add to 00_main.tex with: \\input{06_appendix_qualitative}')